# Практика 1. Первое исследование на финансовых данных
**Количественные финансы · НИУ ВШЭ · 5 ноября 2026**

Сегодня сравниваем два исторических ряда, проверяем данные и учимся объяснять график.
Основная часть рассчитана на 80 минут. Обязательные задания: 1–3. Дополнительное задание – по желанию.

Сохраните свою копию, заполните имя и группу. Здесь нет токенов, API и персональных данных.
Книга: https://quantviews.github.io/quant-finance-book/

**Имя и группа:** …

## 1. Как работать с ноутбуком
Ячейка может содержать текст или код. **Shift + Enter** запускает текущую ячейку.
Выполняйте код сверху вниз. После изменения первой даты нужно повторить все зависимые расчёты.

Ноутбук открывается в Jupyter или Google Colab. Данные загружаются по ссылке из репозитория книги на GitHub. Без интернета ноутбук читает файл `data/prices_2021_2026.csv`, если он лежит рядом с ноутбуком (он есть в архиве занятия).
Для локальной работы заранее установите `pandas`, `matplotlib` и `jupyter`.
Если ячейка завершилась ошибкой, прочитайте последнюю строку сообщения и проверьте, выполнены ли предыдущие ячейки.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Библиотеки загружены")

## 2. Подготовленный снимок
Дневные наблюдения за 2021–2026 годы. Точная первая и последняя даты будут видны в таблице.
Снимок занятия выгружен 05.11.2026, последняя доступная дата – 03.11.2026. 2026 год представлен не полностью. Источник: Московская биржа, поправки на дивиденды – расчёт автора. Снимок свежее данных книги, поэтому числа могут отличаться от примеров в главах.

- `date` – дата наблюдения.
- `SBER`, `LKOH` – `adj_close`, ряды с поправкой на сплиты и дивиденды.
- `SBER_close`, `LKOH_close` – цены с поправкой на сплиты, без дивидендной корректировки.

SBER: ПАО «Сбербанк», ИНН 7707083893. LKOH: ПАО «ЛУКОЙЛ», ИНН 7708004767.

`adj_close` подходит для сравнения относительных изменений. Его уровень не является исторической ценой покупки. Поправка на дивиденды приближает реинвестирование до налогов и издержек.

**Следующая ячейка загружает данные. Выполните её один раз, редактировать не нужно.**

In [ ]:
# Загрузка данных: сначала с GitHub, без сети – из файла рядом с ноутбуком.
import hashlib, io, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/quantviews/quant-finance-book/main/lessons/01-introduction/data/prices_2021_2026.csv"
LOCAL = Path("data/prices_2021_2026.csv")
try:
    raw = urllib.request.urlopen(URL, timeout=20).read()
    print("Данные загружены с GitHub")
except OSError:
    raw = LOCAL.read_bytes()
    print("Нет доступа к GitHub, данные прочитаны из файла", LOCAL)
assert hashlib.sha256(raw).hexdigest() == "90bf1f088434b60be0ea484f521b6eeff95bfeaa9817c6cf4c28656fce6082c9", "Файл отличается от снимка занятия"
df = pd.read_csv(io.BytesIO(raw), parse_dates=["date"])
print("Строк загружено:", len(df))

In [ ]:
df.head()

### Задание 1. Сначала проверьте таблицу
Определите число строк, первую и последнюю даты, наличие пропусков и повторяющихся дат.
Объясните, почему здесь нет всех календарных дней. Можно ли только по отсутствующей дате понять причину отсутствия торгов?

Используйте результат следующей ячейки. **Ваш вывод:** …

In [ ]:
print("Строк:", len(df))
print("Первая дата:", df["date"].min().date())
print("Последняя дата:", df["date"].max().date())
print("Повторяющихся дат:", df["date"].duplicated().sum())
print("Пропусков по столбцам:")
print(df.isna().sum())

## 3. Один график – одна понятная величина
Сделаем даты индексом таблицы. До запуска предскажите: что означает деление на первую строку?

После нормирования обе линии начинаются со 100. Высота линии показывает относительное изменение от начала выбранного периода, а не цену акции в рублях.

In [ ]:
prices = df.set_index("date")[["SBER", "LKOH"]].sort_index()
normalized = 100 * prices / prices.iloc[0]
ax = normalized.plot(color=["#34699A", "#C8702A"])
ax.set(title="Сравнение скорректированных рядов", xlabel="Дата",
       ylabel="Начальная величина = 100")
ax.grid(alpha=0.25)
plt.show()

## 4. Разобранный пример: изменение SBER за весь период
Изменение за период = последнее значение / первое значение − 1.

`.iloc[0]` выбирает первую строку, `.iloc[-1]` – последнюю. В расчётах доля 0.10 означает 10%. Формат `:.1%` меняет только отображение.
Полученная величина относится ко всему периоду, а не к одному году.

In [ ]:
ticker = "SBER"
first = prices[ticker].iloc[0]
last = prices[ticker].iloc[-1]
period_return = last / first - 1
print(f"{ticker}: изменение за весь период {period_return:.1%}")

### Задание 2. Повторите расчёт для LKOH
1. Посчитайте изменение LKOH за тот же период.
2. Сравните с результатом SBER и положением линий в конце графика.
3. Объясните, почему из этого не следует, что LKOH обязательно будет лучше в будущем.

**Ваш вывод:** …

In [ ]:
# Напишите свой расчёт для LKOH.
# Подсказка: используйте пример выше и замените тикер.


### Задание 3. Меняем начальную дату
Сначала предположите, сохранится ли порядок результатов.
Ниже выделен период с 2023 года. Постройте заново нормированные линии и повторите сравнение двух акций.

**Важно:** первая строка для нормирования теперь должна принадлежать новой выборке.

**Ваше предположение:** …

**Что получилось и как изменился вывод:** …

In [ ]:
start = "2023-01-01"
subset = prices.loc[start:]
print("Фактическая первая дата:", subset.index.min().date())
# Допишите нормирование, график и изменение каждой серии за новый период.


## 5. Три предложения вместо подписи «всё выросло»
Завершите вывод:
1. На периоде … мы сравнили … и получили …
2. После изменения начальной даты …
3. Этот расчёт не учитывает … и не позволяет утверждать …

Если использовали ИИ, запишите: **что спросили**, **что получили**, **как проверили**.

**Мой текст:** …

### Дополнительно. Падение от достигнутого максимума
Как мог бы чувствовать себя инвестор, купивший бумагу возле предыдущего максимума?
Для каждой даты нужен максимум только по прошлой истории, включая текущую дату.
Подсказка: `.cummax()` вычисляет такой максимум. Просадка = текущее значение / накопленный максимум − 1.

Найдите минимальную просадку каждого ряда и её дату. Поясните, почему это другая величина, чем изменение от начала выборки.

**Ваш вывод:** …

In [ ]:
# Необязательное задание: рассчитайте просадку и дату её минимума.


## Перед сдачей
- Перезапустите ядро и выполните все ячейки сверху вниз.
- Проверьте, что обязательные задания 1–3 выполнены, а выводы написаны своими словами.
- Сохраните файл `practice01_фамилия.ipynb` и сдайте способом, который объявит преподаватель.
- Не включайте в файл пароли, токены или личные финансовые данные.

**К следующему четвергу, 12 ноября:** завершить задания, прочитать введение и начало главы 2 книги. Если Python для вас новый, разберите каждую строку кода этой практики: что она делает и что выводит.